# Explainability (Grad-CAM & Attention Maps)


## 1. Install pytorch-grad-cam 

In [ ]:
#!pip install grad-cam


## 2. Configuration

In [2]:
from pathlib import Path
import torch

PROJECT_ROOT = Path(r"D:\Ravishi\MSc Final Project\skin-lesion-xai")

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
XAI_DIR = RESULTS_DIR / "explainability"
XAI_DIR.mkdir(parents=True, exist_ok=True)

EVAL_CSV = DATA_PROCESSED / "val.csv"

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

MODEL_CONFIGS = {
    "cnn_baseline":  {"timm_name": "resnet50",             "description": "Baseline CNN (ResNet-50)"},
    "attention_cnn": {"timm_name": "resnet50",             "description": "ResNet-50 + CBAM"},
    "vit":           {"timm_name": "deit_small_patch16_224","description": "DeiT-Small (transformer)"},
}
DISPLAY_NAMES = {"cnn_baseline": "CNN", "attention_cnn": "Attention-CNN", "vit": "DeiT"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 3. Model definitions & loading

In [3]:
import timm
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1); self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False),
                                 nn.ReLU(inplace=True), nn.Conv2d(hidden, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)
    def forward(self, x):
        return self.spatial_attn(self.channel_attn(x))

class AttentionCNN(nn.Module):
    def __init__(self, backbone_name="resnet50", num_classes=2, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.cbam = CBAM(feat_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        feats = self.cbam(self.backbone(x))
        return self.fc(self.dropout(self.pool(feats).flatten(1)))

class DropoutCNN(nn.Module):

    def __init__(self, backbone_name="resnet50", num_classes=2, pretrained=False,
                 dropout=0.5):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, global_pool="")
        feat_dim = self.backbone.num_features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        return self.fc(self.dropout(self.pool(self.backbone(x)).flatten(1)))

def build_model(model_key, num_classes=2):
    timm_name = MODEL_CONFIGS[model_key]["timm_name"]
    if model_key == "attention_cnn":
        return AttentionCNN(timm_name, num_classes, pretrained=False)
    if model_key == "cnn_baseline":
        return DropoutCNN(timm_name, num_classes, pretrained=False)
    return timm.create_model(timm_name, pretrained=False, num_classes=num_classes)

def load_trained(model_key):
    ckpt_path = MODELS_DIR / f"{model_key}_best.pth"
    model = build_model(model_key)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt["model_state"] if "model_state" in ckpt else ckpt
    model.load_state_dict(state)
    model.to(device).eval()
    return model

print("Model definitions ready.")


## 4. Grad-CAM setup per architecture

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def deit_reshape_transform(tensor, height=14, width=14):
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    return result.permute(0, 3, 1, 2)  

def get_cam(model_key, model):
    if model_key == "cnn_baseline":
        target_layers = [model.backbone.layer4[-1]]
        reshape = None
    elif model_key == "attention_cnn":
        target_layers = [model.backbone.layer4[-1]]
        reshape = None
    elif model_key == "vit":
        target_layers = [model.blocks[-1].norm1]
        reshape = deit_reshape_transform
    else:
        raise ValueError(model_key)
    return GradCAM(model=model, target_layers=target_layers, reshape_transform=reshape)

print("Grad-CAM setup ready.")


## 5. Image loading & overlay helpers

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms
from pytorch_grad_cam.utils.image import show_cam_on_image

df = pd.read_csv(EVAL_CSV)

preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_image(path):
    pil = Image.open(path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    rgb = np.array(pil).astype(np.float32) / 255.0
    tensor = preprocess(Image.open(path).convert("RGB")).unsqueeze(0)
    return tensor, rgb

def cam_overlay(cam, input_tensor, rgb, target_class=1):
    grayscale = cam(input_tensor=input_tensor.to(device),
                    targets=[ClassifierOutputTarget(target_class)])[0]
    return show_cam_on_image(rgb, grayscale, use_rgb=True)

print(f"Loaded {len(df)} validation rows.")


## 6. Qualitative grid 


In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
mel_rows = df[df.label == 1].sample(3, random_state=42)
ben_rows = df[df.label == 0].sample(3, random_state=42)
examples = pd.concat([mel_rows, ben_rows]).reset_index(drop=True)

# Preload models and CAMs
models = {k: load_trained(k) for k in MODEL_CONFIGS}
cams = {k: get_cam(k, models[k]) for k in MODEL_CONFIGS}

n = len(examples)
fig, axes = plt.subplots(n, 4, figsize=(14, 3.2 * n))
col_titles = ["Original"] + [DISPLAY_NAMES[k] for k in MODEL_CONFIGS]

for i, (_, row) in enumerate(examples.iterrows()):
    tensor, rgb = load_image(row["image_path"])
    true_label = "MELANOMA" if row["label"] == 1 else "benign"
    axes[i, 0].imshow(rgb)
    axes[i, 0].set_ylabel(true_label, fontsize=11,
                          color="crimson" if row["label"] == 1 else "black")
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
    for j, key in enumerate(MODEL_CONFIGS, start=1):
        overlay = cam_overlay(cams[key], tensor, rgb, target_class=1)
        axes[i, j].imshow(overlay)
        axes[i, j].axis("off")

for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=12)

fig.suptitle("Grad-CAM: where each model looks (melanoma class)", fontsize=13)
plt.tight_layout()
plt.savefig(XAI_DIR / "gradcam_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {XAI_DIR / 'gradcam_grid.png'}")


## 7. False-positive analysis


In [ ]:
cnn = models["cnn_baseline"]
cnn_cam = cams["cnn_baseline"]

fp_examples = []
with torch.no_grad():
    for _, row in df[df.label == 0].iterrows():
        tensor, rgb = load_image(row["image_path"])
        prob = torch.softmax(cnn(tensor.to(device)).float(), dim=1)[0, 1].item()
        if prob >= 0.5:                    
            fp_examples.append((row["image_path"], prob, rgb, tensor))
        if len(fp_examples) >= 4:
            break

if not fp_examples:
    print("No false positives found in the scan window.")
else:
    fig, axes = plt.subplots(2, len(fp_examples), figsize=(4 * len(fp_examples), 8))
    if len(fp_examples) == 1:
        axes = axes.reshape(2, 1)
    for j, (path, prob, rgb, tensor) in enumerate(fp_examples):
        overlay = cam_overlay(cnn_cam, tensor, rgb, target_class=1)
        axes[0, j].imshow(rgb); axes[0, j].axis("off")
        axes[0, j].set_title(f"benign\nCNN said melanoma ({prob:.2f})", fontsize=10)
        axes[1, j].imshow(overlay); axes[1, j].axis("off")
        axes[1, j].set_title("where the CNN looked", fontsize=10)
    fig.suptitle("False positives: CNN wrongly flags benign lesions as melanoma", fontsize=13)
    plt.tight_layout()
    plt.savefig(XAI_DIR / "cnn_false_positives.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {XAI_DIR / 'cnn_false_positives.png'}")
